# Track 5 — Robust Detection of AI-Generated Images

End-to-end run on Kaggle (Accelerator = **GPU T4 x2**, Internet = **On**).

Reproduces every number in the README: the clean-trained baseline, the
augmentation-trained robust model, the 15-condition robustness table, the
full-resolution out-of-distribution check, and the required prediction script.

The code is device-agnostic (`--device auto`), so the same commands run on CPU,
CUDA or Apple MPS. The reported runs were done on an M2 laptop; on a T4 they are
considerably faster.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
# Clone and install
!git clone https://github.com/<you>/<repo>.git /kaggle/working/tj
%cd /kaggle/working/tj
!pip install -q -r requirements.txt

In [ ]:
# Data. Either download the HF mirror (needs Internet = On) ...
!python -m src.get_data

# ... or, if the Kaggle dataset is attached via Add Input, convert it instead:
# !python -m src.get_data --kaggle-dir /kaggle/input/cifake-real-and-ai-generated-synthetic-images

In [ ]:
# Verify the measuring instrument BEFORE training anything.
!python -m src.selftest

In [ ]:
# The core experiment: two runs differing in exactly one variable.
!python -m src.train --aug none   --tag baseline --epochs 12
!python -m src.train --aug robust --tag robust   --epochs 12

In [ ]:
# Robustness table across all 15 conditions, on the held-out 20k test split.
!python -m src.eval_robustness --checkpoints baseline robust --save-scores

In [ ]:
# Error analysis (needs --save-scores above).
!python -m src.error_analysis

In [ ]:
# Full-resolution out-of-distribution check: 600 SID_Set validation images at ~1024px.
!python -m src.fetch_fullres --per-class 300
!python -m src.eval_fullres --checkpoints baseline robust --modes patch resize

In [ ]:
# Required deliverable: image directory -> JSON of AI-generated confidences.
!python -m src.predict --image-dir data/fullres/ai --output results/preds.json
!head -20 results/preds.json